# **SPRSound** Dataset Preparation for Advanced Models

## 1. Set up Imports

In [11]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

# Add project root
sys.path.append('..')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Load the enriched features

In [12]:
df_all = pd.read_csv("../../sound_data/features/sprsound_augmented_features.csv")


print("="*60)
print("FEATURES LOADED")
print("="*60)

print(f"Loaded features: {df_all.shape}")
print(f"Columns: {list(df_all.columns)[:10]}...")

df_all.head()

# --- QUICK FIX: Reconstruct 'is_augmented' if it's missing ---
if 'is_augmented' not in df_all.columns:
    # If the filename contains '_aug_', it's synthetic (1), otherwise it's real (0)
    df_all['is_augmented'] = df_all['filename'].apply(lambda x: 1 if '_aug_' in str(x) else 0)
    print(f"Reconstructed 'is_augmented' column. Found {df_all['is_augmented'].sum()} augmented files.")

FEATURES LOADED
Loaded features: (20687, 37)
Columns: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std']...


## 3. Extract Patient IDs

In [13]:
# Because augmented files were named 'PatientID_..._aug_noise.wav', 
# splitting by '_' perfectly groups augmented files with their original patient!
df_all['patient_id'] = df_all['filename'].apply(lambda x: str(x).split('_')[0])
print(f"Total unique patients: {df_all['patient_id'].nunique()}")

# 3. Patient-level Split (70% Train, 15% Val, 15% Test)
# This guarantees no patient in the test set was ever seen during training
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss1.split(df_all, groups=df_all['patient_id']))

df_train_all = df_all.iloc[train_idx]
df_temp = df_all.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['patient_id']))

df_val_all = df_temp.iloc[val_idx]
df_test_all = df_temp.iloc[test_idx]

Total unique patients: 869


## 4. Purge synthetic data from Evaluation Sets

In [14]:
# We train on everything, but we validate and test ONLY on real, original audio
df_train = df_train_all.copy() 
df_val = df_val_all[df_val_all['is_augmented'] == 0].copy()
df_test = df_test_all[df_test_all['is_augmented'] == 0].copy()

print("\n" + "="*50)
print("FINAL DATA SPLIT (Real + Augmented)")
print("="*50)
print(f"Train Set (Includes Aug): {len(df_train)} samples")
print(f"Val Set   (Real ONLY):    {len(df_val)} samples")
print(f"Test Set  (Real ONLY):    {len(df_test)} samples")


FINAL DATA SPLIT (Real + Augmented)
Train Set (Includes Aug): 13232 samples
Val Set   (Real ONLY):    1866 samples
Test Set  (Real ONLY):    2889 samples


## 5. Select Feature Labels

In [15]:
# Select feature columns (exclude metadata columns)
feature_cols = [c for c in df_all.columns if c.startswith(('mfcc', 'zcr', 'rms', 'harmonic', 'spectral', 'duration'))]

print(f"\n Selected {len(feature_cols)} features")
print(f"First 10 features: {feature_cols[:10]}")


 Selected 33 features
First 10 features: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std']
